# Stage 3.1: Improving and comparing my RAG retriever

**Project:** Agentic AI system for automated research and report generation  
**Application topic:** AI in healthcare  
**Student:** Adarsh Konderu

Stage 3 confirmed that dense semantic retrieval works, but it also showed that
a broadly related passage can sometimes rank above a passage containing the
exact technical concept. This notebook therefore compares:

- **Baseline:** dense semantic similarity only;
- **Improved version:** dense similarity + BM25 keyword matching + source
  diversity.

Both retrievers use the same chunks, embedding model and fixed test questions.
This makes the comparison fair and reproducible.


## 1. Why use hybrid retrieval?

Dense retrieval is useful when the query and evidence express the same meaning
with different words. BM25 is useful when exact terms such as *hallucination*,
*bias* or *resource allocation* are important.

I combine their rankings using weighted Reciprocal Rank Fusion (RRF):

`hybrid score = 0.65 / (60 + dense rank) + 0.35 / (60 + BM25 rank)`

The weights give more importance to semantic meaning while still rewarding
exact terminology. The hybrid output is also restricted to one chunk from each
article in the first five results to improve source diversity.


In [ ]:
# The same public embedding model used in Stage 3 is needed for query vectors.
!pip -q install sentence-transformers


## 2. Import libraries and fix the experiment settings

The values below are recorded before testing. Changing them after seeing the
results would make the comparison less reliable.


In [ ]:
import csv
import importlib.metadata
import io
import json
import math
import re
import shutil
import sys
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from google.colab import files
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path("/content/stage_3_1_input")
OUTPUT_DIR = Path("/content/stage_3_1_outputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 5
BASELINE_MAXIMUM_PER_ARTICLE = 2
HYBRID_MAXIMUM_PER_ARTICLE = 1
RRF_CONSTANT = 60
DENSE_WEIGHT = 0.65
BM25_WEIGHT = 0.35
BM25_K1 = 1.5
BM25_B = 0.75

print("Dense weight:", DENSE_WEIGHT)
print("BM25 weight:", BM25_WEIGHT)
print("Hybrid source limit:", HYBRID_MAXIMUM_PER_ARTICLE, "chunk per article")


## 3. Upload the Stage 3 retrieval evidence

Upload `Adarsh_Konderu_Stage_3_Retrieval_Evidence.zip`. This reuses the 1,450
chunks and their embeddings, so the complete article corpus does not need to be
processed again.


In [ ]:
uploaded_files = files.upload()

expected_name = "Adarsh_Konderu_Stage_3_Retrieval_Evidence.zip"
if expected_name in uploaded_files:
    uploaded_name = expected_name
elif len(uploaded_files) == 1:
    uploaded_name = next(iter(uploaded_files))
else:
    raise ValueError("Please upload only the Stage 3 retrieval evidence ZIP.")

archive_bytes = uploaded_files[uploaded_name]

# Check archive member paths before extraction.
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for member in archive.namelist():
        member_path = Path(member)
        if member_path.is_absolute() or ".." in member_path.parts:
            raise ValueError(f"Unsafe archive path: {member}")
    archive.extractall(INPUT_DIR)

required_files = [
    "pmc_chunks.jsonl",
    "pmc_chunk_embeddings.npy",
    "retrieval_run_metadata.json",
]
for required_file in required_files:
    assert (INPUT_DIR / required_file).exists(), f"Missing {required_file}"

print("Stage 3 evidence extracted correctly")


## 4. Load and verify the Stage 3 chunks and embeddings

The integrity checks ensure that the hybrid experiment uses exactly the same
1,450 chunks and 384-dimensional vectors as the baseline experiment.


In [ ]:
chunks = [
    json.loads(line)
    for line in (INPUT_DIR / "pmc_chunks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
embeddings = np.load(INPUT_DIR / "pmc_chunk_embeddings.npy")
stage_3_metadata = json.loads(
    (INPUT_DIR / "retrieval_run_metadata.json").read_text(encoding="utf-8")
)

assert len(chunks) == 1450
assert embeddings.shape == (1450, 384)
assert len({chunk["chunk_id"] for chunk in chunks}) == 1450
assert len({chunk["pmcid"] for chunk in chunks}) == 46
assert np.isfinite(embeddings).all()
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5)
assert stage_3_metadata["embedding_model"] == MODEL_NAME

print("Chunks:", len(chunks))
print("Articles:", len({chunk["pmcid"] for chunk in chunks}))
print("Embedding matrix:", embeddings.shape)


## 5. Build a transparent BM25 keyword index

BM25 rewards important query words while reducing the influence of very common
words. The implementation is shown directly rather than hidden inside a search
framework, so each part can be explained.


In [ ]:
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "being", "by",
    "can", "could", "did", "do", "does", "for", "from", "had", "has",
    "have", "how", "in", "into", "is", "it", "its", "may", "of", "on",
    "or", "should", "that", "the", "their", "these", "this", "to", "use",
    "used", "using", "was", "were", "what", "when", "where", "which",
    "with", "would",
}
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)?")


def tokenize(text):
    """Create lower-case BM25 terms and remove common English words."""
    return [
        token
        for token in TOKEN_PATTERN.findall(text.lower())
        if token not in STOP_WORDS and len(token) >= 2
    ]


# The title and theme are included because they provide useful document context.
document_tokens = [
    tokenize(
        f'{chunk["title"]} {chunk["theme"].replace("_", " ")} {chunk["text"]}'
    )
    for chunk in chunks
]
document_lengths = np.array([len(tokens) for tokens in document_tokens], dtype=np.float32)
average_document_length = float(document_lengths.mean())

term_frequencies = [Counter(tokens) for tokens in document_tokens]
document_frequency = Counter()
for frequencies in term_frequencies:
    document_frequency.update(frequencies.keys())

document_count = len(chunks)
inverse_document_frequency = {
    term: math.log(1 + (document_count - frequency + 0.5) / (frequency + 0.5))
    for term, frequency in document_frequency.items()
}


def calculate_bm25_scores(question):
    """Calculate one BM25 relevance score for every chunk."""
    query_terms = tokenize(question)
    scores = np.zeros(document_count, dtype=np.float32)

    for document_index, frequencies in enumerate(term_frequencies):
        length_adjustment = BM25_K1 * (
            1 - BM25_B + BM25_B * document_lengths[document_index] / average_document_length
        )
        score = 0.0
        for term in query_terms:
            frequency = frequencies.get(term, 0)
            if frequency == 0:
                continue
            score += inverse_document_frequency.get(term, 0.0) * (
                frequency * (BM25_K1 + 1) / (frequency + length_adjustment)
            )
        scores[document_index] = score

    return scores


print("BM25 index created for", document_count, "chunks")
print("Unique BM25 terms:", len(document_frequency))


## 6. Define the baseline and hybrid retrievers

The baseline ranks only by cosine similarity. The hybrid method fuses the dense
and BM25 ranks. Results retain both component ranks so the final order can be
audited.


In [ ]:
embedding_model = SentenceTransformer(MODEL_NAME)


def create_rank_positions(order):
    """Convert an ordered list of indexes into one-based rank positions."""
    positions = np.empty(len(order), dtype=np.int32)
    positions[order] = np.arange(1, len(order) + 1)
    return positions


def select_with_source_limit(order, maximum_per_article, top_k):
    """Select results while limiting repeated chunks from one article."""
    selected = []
    counts = Counter()
    for index in order:
        article_id = chunks[int(index)]["pmcid"]
        if counts[article_id] >= maximum_per_article:
            continue
        selected.append(int(index))
        counts[article_id] += 1
        if len(selected) == top_k:
            break
    return selected


def retrieve_both_methods(question, top_k=TOP_K):
    """Run the dense baseline and improved hybrid method for one question."""
    question_embedding = embedding_model.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    )[0]
    dense_scores = embeddings @ question_embedding
    bm25_scores = calculate_bm25_scores(question)

    dense_order = np.argsort(-dense_scores)
    bm25_order = np.argsort(-bm25_scores)
    dense_ranks = create_rank_positions(dense_order)
    bm25_ranks = create_rank_positions(bm25_order)

    hybrid_scores = (
        DENSE_WEIGHT / (RRF_CONSTANT + dense_ranks)
        + BM25_WEIGHT / (RRF_CONSTANT + bm25_ranks)
    )
    hybrid_order = np.argsort(-hybrid_scores)

    baseline_indexes = select_with_source_limit(
        dense_order, BASELINE_MAXIMUM_PER_ARTICLE, top_k
    )
    hybrid_indexes = select_with_source_limit(
        hybrid_order, HYBRID_MAXIMUM_PER_ARTICLE, top_k
    )

    def format_results(indexes, method):
        formatted = []
        for rank, index in enumerate(indexes, start=1):
            result = dict(chunks[index])
            result.update(
                {
                    "method": method,
                    "rank": rank,
                    "dense_similarity": round(float(dense_scores[index]), 6),
                    "bm25_score": round(float(bm25_scores[index]), 6),
                    "dense_rank": int(dense_ranks[index]),
                    "bm25_rank": int(bm25_ranks[index]),
                    "hybrid_rrf_score": round(float(hybrid_scores[index]), 9),
                }
            )
            formatted.append(result)
        return formatted

    return (
        format_results(baseline_indexes, "dense_baseline"),
        format_results(hybrid_indexes, "hybrid_rrf"),
    )


## 7. Use ten fixed development questions

The original five questions are retained, and five further questions provide a
second example for each broad area. The questions are fixed before comparing
the two retrievers.


In [ ]:
TEST_QUERIES = [
    {
        "query_id": "Q1",
        "question": "How is artificial intelligence used to support medical imaging diagnosis?",
        "expected_themes": ["medical_imaging_and_diagnosis"],
    },
    {
        "query_id": "Q2",
        "question": "How does AI support clinical decisions and treatment recommendations?",
        "expected_themes": ["clinical_decision_support"],
    },
    {
        "query_id": "Q3",
        "question": "How can artificial intelligence improve healthcare operations and clinical workflows?",
        "expected_themes": ["healthcare_operations"],
    },
    {
        "query_id": "Q4",
        "question": "What hallucination and factual accuracy risks occur when healthcare reports use large language models?",
        "expected_themes": ["generative_ai_and_llms", "ethics_safety_and_bias"],
    },
    {
        "query_id": "Q5",
        "question": "What ethical, safety and bias limitations are reported for AI in healthcare?",
        "expected_themes": ["ethics_safety_and_bias"],
    },
    {
        "query_id": "Q6",
        "question": "How do deep learning systems analyse radiology and ultrasound images for disease detection?",
        "expected_themes": ["medical_imaging_and_diagnosis"],
    },
    {
        "query_id": "Q7",
        "question": "Why is human oversight important when AI recommends a diagnosis or treatment?",
        "expected_themes": ["clinical_decision_support", "ethics_safety_and_bias"],
    },
    {
        "query_id": "Q8",
        "question": "How can AI improve hospital administration, resource allocation and workflow efficiency?",
        "expected_themes": ["healthcare_operations"],
    },
    {
        "query_id": "Q9",
        "question": "What citation, misinformation and fabricated-reference problems affect LLM scientific writing in healthcare?",
        "expected_themes": ["generative_ai_and_llms", "ethics_safety_and_bias"],
    },
    {
        "query_id": "Q10",
        "question": "How can biased healthcare AI discriminate against underrepresented patient groups?",
        "expected_themes": ["ethics_safety_and_bias"],
    },
]

print("Fixed development questions:", len(TEST_QUERIES))


## 8. Run both retrievers and calculate development metrics

`Theme match at 5` is retained only as a coarse sanity check. `Source diversity
at 5` reports the proportion of different PMC articles in the five results.
Neither metric is treated as the final dissertation evaluation.


In [ ]:
all_result_rows = []
comparison_rows = []

for test_query in TEST_QUERIES:
    baseline_results, hybrid_results = retrieve_both_methods(test_query["question"])
    expected_themes = set(test_query["expected_themes"])

    for method, results in [
        ("dense_baseline", baseline_results),
        ("hybrid_rrf", hybrid_results),
    ]:
        theme_matches = sum(result["theme"] in expected_themes for result in results)
        unique_sources = len({result["pmcid"] for result in results})

        comparison_rows.append(
            {
                "query_id": test_query["query_id"],
                "question": test_query["question"],
                "method": method,
                "expected_themes": " | ".join(test_query["expected_themes"]),
                "theme_match_at_5": theme_matches / len(results),
                "source_diversity_at_5": unique_sources / len(results),
                "rank_1_expected_theme": int(results[0]["theme"] in expected_themes),
            }
        )

        for result in results:
            all_result_rows.append(
                {
                    "query_id": test_query["query_id"],
                    "question": test_query["question"],
                    "expected_themes": " | ".join(test_query["expected_themes"]),
                    "method": method,
                    "rank": result["rank"],
                    "chunk_id": result["chunk_id"],
                    "pmcid": result["pmcid"],
                    "theme": result["theme"],
                    "title": result["title"],
                    "source_url": result["source_url"],
                    "dense_similarity": result["dense_similarity"],
                    "bm25_score": result["bm25_score"],
                    "dense_rank": result["dense_rank"],
                    "bm25_rank": result["bm25_rank"],
                    "hybrid_rrf_score": result["hybrid_rrf_score"],
                    "text_preview": result["text"][:700],
                }
            )


def mean_metric(method, metric):
    values = [
        float(row[metric]) for row in comparison_rows if row["method"] == method
    ]
    return sum(values) / len(values)


for method in ["dense_baseline", "hybrid_rrf"]:
    print("\nMethod:", method)
    print("Mean theme match at 5:", round(mean_metric(method, "theme_match_at_5"), 3))
    print("Mean source diversity at 5:", round(mean_metric(method, "source_diversity_at_5"), 3))
    print("Rank 1 expected-theme rate:", round(mean_metric(method, "rank_1_expected_theme"), 3))


## 9. Inspect how the difficult hallucination query changed

The output below shows whether BM25 promoted passages that explicitly mention
hallucination, misinformation or factual accuracy.


In [ ]:
for method in ["dense_baseline", "hybrid_rrf"]:
    print("\n", "=" * 80)
    print("Q4 method:", method)
    method_rows = [
        row
        for row in all_result_rows
        if row["query_id"] == "Q4" and row["method"] == method
    ]
    for row in method_rows[:3]:
        print("\nRank", row["rank"], "PMCID", row["pmcid"])
        print("Title:", row["title"])
        print("Dense rank:", row["dense_rank"], "BM25 rank:", row["bm25_rank"])
        print("Passage:", row["text_preview"][:500], "...")


## 10. Save the comparison evidence

The ZIP contains every ranked result, the per-query comparison, fixed questions
and run settings. We will inspect these files before choosing a retriever for
the planner, writer and reviewer agents.


In [ ]:
def write_csv(path, rows):
    """Write a non-empty list of dictionaries to CSV."""
    with path.open("w", encoding="utf-8", newline="") as file_handle:
        writer = csv.DictWriter(file_handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


write_csv(OUTPUT_DIR / "dense_and_hybrid_results.csv", all_result_rows)
write_csv(OUTPUT_DIR / "retriever_comparison_summary.csv", comparison_rows)
(OUTPUT_DIR / "fixed_development_queries.json").write_text(
    json.dumps(TEST_QUERIES, indent=2), encoding="utf-8"
)

run_metadata = {
    "stage": "Stage 3.1 dense versus hybrid retrieval comparison",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "sentence_transformers_version": importlib.metadata.version("sentence-transformers"),
    "numpy_version": np.__version__,
    "embedding_model": MODEL_NAME,
    "chunk_count": len(chunks),
    "article_count": len({chunk["pmcid"] for chunk in chunks}),
    "query_count": len(TEST_QUERIES),
    "top_k": TOP_K,
    "baseline_maximum_per_article": BASELINE_MAXIMUM_PER_ARTICLE,
    "hybrid_maximum_per_article": HYBRID_MAXIMUM_PER_ARTICLE,
    "rrf_constant": RRF_CONSTANT,
    "dense_weight": DENSE_WEIGHT,
    "bm25_weight": BM25_WEIGHT,
    "bm25_k1": BM25_K1,
    "bm25_b": BM25_B,
    "framework": "Direct Python implementation; no LangChain",
    "metric_warning": (
        "Theme match is a coarse article-label sanity check, not final relevance precision."
    ),
}
(OUTPUT_DIR / "stage_3_1_run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2), encoding="utf-8"
)

archive_path = shutil.make_archive(
    "/content/Adarsh_Konderu_Stage_3_1_Hybrid_Retrieval_Evidence",
    "zip",
    OUTPUT_DIR,
)

print("Comparison rows:", len(comparison_rows))
print("Ranked result rows:", len(all_result_rows))
print("Evidence package:", archive_path)
files.download(archive_path)


## 11. How I will interpret this comparison

I will not select the hybrid method only because one automatic number is
higher. I will inspect whether it retrieves more direct evidence, whether its
sources are more diverse and whether the citations remain traceable.

This remains development testing. The final dissertation evaluation will use a
larger fixed question set, explicit scoring criteria and multiple LLM judges.
